# t-SNE эмбеддингов

Как t-SNE эмбеддингов токенов в Image-Transform-Predict, только точки здесь картинки теста: реальные из разных
источников, каждый генератор, полные генерации и локальные правки SID, отрисованные тексты и логотипы.
Эмбеддинг картинки: признаки головы детектора перед последним слоем (256 чисел на патч), усреднённые по патчам.
Генератор, который лёг рядом с реальными, модель путает с ними.

In [ ]:
import os, io, json, random, sys
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())

from collections import defaultdict
from pathlib import Path
import numpy as np
import yaml, torch
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.manifold import TSNE
from tqdm import tqdm
from src.data import load_rows
from src.model import Detector

cfg = yaml.safe_load(open('configs/train.yaml'))
data, out, size = Path(cfg['data']['dir']), Path(cfg['train']['out']), cfg['train']['size']
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = Detector.load(out / 'best.pt', device)


def prepare(path):   # как на тесте, см. predict.ipynb
    image = Image.open(path)
    fmt, image = image.format, image.convert('RGB')
    if fmt != 'JPEG':
        buf = io.BytesIO()
        image.save(buf, 'JPEG', quality=90)
        image = Image.open(buf).convert('RGB')
    image = image.resize((size, size), Image.BICUBIC)
    return torch.from_numpy(np.array(image, dtype=np.float32) / 255).permute(2, 0, 1)


@torch.no_grad()
def embed(path):
    x = prepare(path)[None].to(device)
    with torch.autocast('cuda', dtype=torch.bfloat16, enabled=device == 'cuda'):
        features = model.head[:-1](torch.cat(model.backbone(x), dim=1))   # [1, 256, 32, 32]
    return features.float().mean(dim=(2, 3))[0].cpu().numpy()

In [ ]:
n = 100   # картинок на группу

groups = defaultdict(list)
for r in load_rows(data / 'openfake_test.csv') + load_rows(data / 'sid_validation.csv'):
    groups[r['generator'] or r['source']].append(r['path'])   # реальные по источнику: openfake/docci, sid/real
groups['renders'] = json.load(open(out / 'calibration.json'))['renders_test']

items = [(name, p) for name, paths in sorted(groups.items()) for p in random.Random(0).sample(paths, min(n, len(paths)))]
names = np.array([name for name, _ in items])
vectors = np.stack([embed(p) for _, p in tqdm(items)])
vectors.shape

In [ ]:
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['DejaVu Serif'],
    'font.size': 14,
    'axes.titlesize': 16,
    'grid.alpha': 0.3,
    'figure.dpi': 150,
})

points = TSNE(n_components=2, perplexity=min(30, len(vectors) - 1), random_state=42).fit_transform(vectors)

fig, ax = plt.subplots(figsize=(12, 10))
for i, name in enumerate(sorted(groups)):
    xy = points[names == name]
    ax.scatter(xy[:, 0], xy[:, 1], color=plt.cm.tab20(i % 20), s=25, alpha=0.7, edgecolors='white', linewidths=0.3)
    ax.annotate(name, np.median(xy, axis=0), fontsize=11, xytext=(10, 5), textcoords='offset points',
                bbox=dict(boxstyle='round,pad=0.4', fc='white', alpha=0.9, ec='gray', lw=0.5),
                arrowprops=dict(arrowstyle='->', lw=0.5, color='gray'))

ax.set_title('t-SNE эмбеддингов картинок теста', pad=20)
ax.set_xlabel('t-SNE 1')
ax.set_ylabel('t-SNE 2')
ax.grid(True, linestyle='--')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
# plt.savefig('tsne.pdf', bbox_inches='tight')
plt.show()